# Model Evaluation Notebook

This notebook demonstrates how to run the model evaluation from the `eval_multiple.py` script within a Jupyter environment.

In [ ]:
from typing import Any, Dict, List, Tuple

import hydra
import rootutils
from lightning import LightningDataModule, LightningModule, Trainer
from omegaconf import DictConfig, OmegaConf
import glob
import os
import numpy as np
import torch
import yaml

# Set environment variables and precision
os.environ['HYDRA_FULL_ERROR'] = '1'
torch.set_float32_matmul_precision('highest')

# Setup root directory for the project
root_dir = rootutils.find_root(search_from=os.path.dirname(os.getcwd()), indicator=".project-root")
rootutils.setup_root(root_dir, indicator=".project-root", pythonpath=True)

from src.utils import (
    RankedLogger,
    extras,
    instantiate_loggers,
    log_hyperparameters,
    task_wrapper,
)
from src.utils.metrics import calculate_metrics

In [ ]:
def mean_logits_per_oid(oids,logits,labels):
    """Mean logits per object id.

    :param dataset: Dataset to use.
    :param all_idx: All idxs.
    :param all_logits: All logits.
    :return: Mean logits per object id.
    """

    #from oids ids by taking the part before _ 
    list_oids = []
    for oid in oids:
        list_oids.append(oid.decode('utf-8').split('_')[0])

    list_oids = np.array(list_oids)
    #create a dictionary with oids,logits,labels
    dict_logits = {}
    for i in range(len(list_oids)):
        if list_oids[i] not in dict_logits:
            dict_logits[list_oids[i]] = {'logits': [], 'labels': []}
        dict_logits[list_oids[i]]['logits'].append(logits[i])
        dict_logits[list_oids[i]]['labels'].append(labels[i])
    #mean logits and labels
    for key in dict_logits:
        dict_logits[key]['logits'] = np.argmax(np.mean(np.array(dict_logits[key]['logits']), axis=0))
        dict_logits[key]['labels'] = np.mean(np.array(dict_logits[key]['labels']), axis=0)
    #create a dictionary with oids,logits,labels
    return dict_logits

    

In [ ]:
from thop import profile
import time

log = RankedLogger(__name__, rank_zero_only=True)


def evaluate(cfg: DictConfig, test_type: str) -> Tuple[Dict[str, Any], Dict[str, Any]]:
    """Evaluates given checkpoint on a datamodule testset.

    :param cfg: DictConfig configuration composed by Hydra.
    :param test_type: Type of test set to evaluate.
    :return: Tuple[dict, dict] with metrics and dict with all instantiated objects.
    """
    assert cfg.ckpt_path, "Checkpoint path must be provided."
    # Disable all loggers by setting the logger config to an empty list or None
    cfg.trainer.logger = False

    cfg.data.test_set_type = test_type
    cfg.data.normalize_tab = True
    cfg.data.return_snids = True
    trainer: Trainer = hydra.utils.instantiate(cfg.trainer)

    log.info("Starting testing!")

    # Find all .ckpt files recursively under cfg.ckpt_path, expecting structure like 0/checkpoints/xxx.ckpt, 1/checkpoints/xxx.ckpt, etc.
    ckpt_files = glob.glob(os.path.join(cfg.ckpt_path, "*", "checkpoints", "*.ckpt"))
    # Remove any checkpoint files that end with 'last.ckpt'
    ckpt_files = [f for f in ckpt_files if not f.endswith('last.ckpt')]

    #filter ckpt by a condition in dirname/.hydra/config.yaml , if   data.percentage >= 1
    ckpt_files_clean = []
    for ckpt in ckpt_files:
        with open(os.path.join(os.path.dirname(os.path.dirname(ckpt)), ".hydra", "config.yaml"), 'r') as f:
            config_yaml = yaml.safe_load(f)
        data_percentage = config_yaml['data'].get('percentage', 1)
        if data_percentage >= 1:
            ckpt_files_clean.append(ckpt)
    
    #data splits
    ckpt_files = ckpt_files_clean
    data_splits = []
    for ckpt in ckpt_files:
        with open(os.path.join(os.path.dirname(os.path.dirname(ckpt)), ".hydra", "config.yaml"), 'r') as f:
            config_yaml = yaml.safe_load(f)
        data_splits.append(config_yaml['data'].get('split', 0))
    print(len(ckpt_files), len(data_splits))

    #sort data splits and ckpt_files accordingly
    data_splits, ckpt_files = zip(*sorted(zip(data_splits, ckpt_files)))

        
        

    all_targets_lcs, all_preds_lcs = [], []
    all_targets_feat, all_preds_feat = [], []
    all_targets_mix, all_preds_mix = [], []
    
    # Flag to compute model complexity only once
    complexity_computed = False
    
    for split, ckpt in enumerate(ckpt_files):
        print(f"Evaluating split {split} with checkpoint {ckpt}")
        cfg.data.split = split % 5
        datamodule: LightningDataModule = hydra.utils.instantiate(cfg.data)
        model: LightningModule = hydra.utils.instantiate(cfg.model)

        # Compute FLOPs, parameters, and inference time only once
        if not complexity_computed:
            # Get a sample input from the datamodule
            datamodule.setup('test')
            sample_batch = next(iter(datamodule.test_dataloader()))
            
            # Move model to CUDA for FLOPs computation (required for Flash Attention)
            device = f'cuda:{cfg.trainer.devices[0]}' if torch.cuda.is_available() else 'cpu'
            model = model.to(device)
            model.eval()
            
            # Prepare input tensor (adjust based on your model's input structure)
            # You may need to modify this based on your model's expected input
            input_tensor = {k: v.to(device) if isinstance(v, torch.Tensor) else v 
                          for k, v in sample_batch.items() if k != 'targets' and k != 'oid'}
            
            # Prepare output file path
            output_file = os.path.join(cfg.paths.output_dir, 'model_complexity.txt')
            
            # Compute FLOPs and parameters
            try:
                flops, params = profile(model, inputs=(input_tensor,), verbose=False)
                flops_g = flops / 1e9
                params_m = params / 1e6
                
                print(f"\n{'='*50}")
                print(f"Model Complexity:")
                print(f"FLOPs: {flops_g:.2f}G")
                print(f"Params: {params_m:.2f}M")
            except Exception as e:
                print(f"Could not compute FLOPs: {e}")
                # Fall back to counting parameters only
                params = sum(p.numel() for p in model.parameters())
                params_m = params / 1e6
                flops_g = 0
                print(f"Params (counted): {params_m:.2f}M")
            
            # Compute inference time
            with torch.no_grad():
                # Warmup
                for _ in range(10):
                    _ = model(input_tensor)
                
                # Measure
                if torch.cuda.is_available():
                    torch.cuda.synchronize()
                
                start = time.time()
                num_iterations = 100
                for _ in range(num_iterations):
                    _ = model(input_tensor)
                
                if torch.cuda.is_available():
                    torch.cuda.synchronize()
                    
                avg_time = (time.time() - start) / num_iterations
                avg_time_ms = avg_time * 1000
                
                print(f"Average Inference Time: {avg_time_ms:.2f}ms")
                print(f"{'='*50}\n")
            
            # Write to file
            with open(output_file, 'w') as f:
                f.write(f"MODEL COMPLEXITY ANALYSIS\n")
                f.write(f"{'='*70}\n\n")
                f.write(f"Experiment: {cfg.get('experiment_name', 'Unknown')}\n")
                f.write(f"Model Path: {cfg.ckpt_path}\n")
                f.write(f"Generated: {time.strftime('%Y-%m-%d %H:%M:%S')}\n\n")
                f.write(f"{'-'*70}\n")
                f.write(f"COMPUTATIONAL COMPLEXITY\n")
                f.write(f"{'-'*70}\n")
                f.write(f"FLOPs: {flops_g:.2f} G\n")
                f.write(f"Parameters: {params_m:.2f} M\n\n")
                f.write(f"{'-'*70}\n")
                f.write(f"INFERENCE PERFORMANCE\n")
                f.write(f"{'-'*70}\n")
                f.write(f"Average Inference Time: {avg_time_ms:.2f} ms\n")
                f.write(f"{'='*70}\n")
            
            print(f"✓ Model complexity saved to: {output_file}\n")
            
            complexity_computed = True
        model: LightningModule = hydra.utils.instantiate(cfg.model)
        out = trainer.predict(model=model, dataloaders=datamodule, ckpt_path=ckpt)

        model_logits_lcs,model_logits_feat,model_logits_mix, model_targets, model_oids = [], [], [], [], []

        for batch in out:
            model_targets.append(batch['targets'])
            model_logits_lcs.append(batch['logits_lc'])
            model_logits_feat.append(batch['logits_feat'])
            model_logits_mix.append(batch['logits_mix'])
            model_oids += batch['oid']
        
        if model_logits_lcs and model_logits_lcs[0] is not None:
            model_logits_lcs = torch.cat(model_logits_lcs, axis=0).to(torch.float32).detach().cpu().numpy()
        else:
            model_logits_lcs = None
        if model_logits_feat and model_logits_feat[0] is not None:
            model_logits_feat = torch.cat(model_logits_feat, axis=0).to(torch.float32).detach().cpu().numpy()
        else:
            model_logits_feat = None
        if model_logits_mix and model_logits_mix[0] is not None:
            model_logits_mix = torch.cat(model_logits_mix, axis=0).to(torch.float32).detach().cpu().numpy()
        else:
            model_logits_mix = None
        model_targets = torch.cat(model_targets, axis=0).detach().cpu().numpy()
        if model_logits_lcs is not None:
            matched_predictions_lcs = mean_logits_per_oid(oids=model_oids, logits=model_logits_lcs, labels=model_targets)
            all_targets_lcs.append(np.array([matched_predictions_lcs[key]['labels'] for key in matched_predictions_lcs]))
            all_preds_lcs.append(np.array([matched_predictions_lcs[key]['logits'] for key in matched_predictions_lcs]))
        else:
            all_preds_lcs = None
        if model_logits_feat is not None:
            matched_predictions_feat = mean_logits_per_oid(oids=model_oids, logits=model_logits_feat, labels=model_targets)
            all_targets_feat.append(np.array([matched_predictions_feat[key]['labels'] for key in matched_predictions_feat]))
            all_preds_feat.append(np.array([matched_predictions_feat[key]['logits'] for key in matched_predictions_feat]))
        else:
            all_preds_feat = None
    
        if model_logits_mix is not None:
            matched_predictions_mix = mean_logits_per_oid(oids=model_oids, logits=model_logits_mix, labels=model_targets)
            all_targets_mix.append(np.array([matched_predictions_mix[key]['labels'] for key in matched_predictions_mix]))
            all_preds_mix.append(np.array([matched_predictions_mix[key]['logits'] for key in matched_predictions_mix]))
        else:
            all_preds_mix = None
    

    if all_preds_lcs is not None:
        cm_title_lcs = cfg.get('cm_title')
        if cm_title_lcs:
            cm_title_lcs = cm_title_lcs.replace('MD-', '').replace('FT-', '')
        

        calculate_metrics(
            targets_list=all_targets_lcs,
            preds_list=all_preds_lcs,
            path=cfg.paths.output_dir,
            classes=cfg.get('classes'),
            classes_order=cfg.get('classes_order'),
            cm_title=cm_title_lcs,
            experiment_name= cfg.get('experiment_name', 'new_atat_windows'),
            all_experiments_csv= cfg.get('all_experiments_csv', None),
            baseline_path=cfg.get('baseline_path', None),
            baseline_experiment_name= cfg.get('baseline_experiment_name', None),
            modality= 'LC'
        )
    if all_preds_feat is not None:
        cm_title_feat = cfg.get('cm_title')
        if cm_title_feat:
            cm_title_feat = cm_title_feat.replace('LC-', '')
        if 'FT' not in cm_title_feat:
            cm_title_feat = cm_title_feat.replace('-MTA', '')
        calculate_metrics(
            targets_list=all_targets_feat,
            preds_list=all_preds_feat,
            path=cfg.paths.output_dir,
            classes=cfg.get('classes'),
            classes_order=cfg.get('classes_order'),
            cm_title=cm_title_feat,
            experiment_name= cfg.get('experiment_name', 'new_atat_windows'),
            all_experiments_csv= cfg.get('all_experiments_csv', None),
            baseline_path=cfg.get('baseline_path', None),
            baseline_experiment_name= cfg.get('baseline_experiment_name', None),
            modality= 'Feat'
        )
    if all_preds_mix is not None:
        calculate_metrics(
            targets_list=all_targets_mix,
            preds_list=all_preds_mix,
            path=cfg.paths.output_dir,
            classes=cfg.get('classes'),
            classes_order=cfg.get('classes_order'),
            cm_title=cfg.get('cm_title'),
            experiment_name= cfg.get('experiment_name', 'new_atat_windows'),
            all_experiments_csv= cfg.get('all_experiments_csv', None),
            baseline_path=cfg.get('baseline_path', None),
            baseline_experiment_name= cfg.get('baseline_experiment_name', None),
            modality= 'Mix'
        )


In [ ]:
def eval_several_exp(exp_dict):
    name = exp_dict['name']
    cm_title = exp_dict['cm_title']
    experiment_name = exp_dict['experiment_name']
    baseline_name = exp_dict['baseline_name']
    type = exp_dict.get('type', 'lc')
    experiment_path = "../logs/" + type + '/' + name
    experiment_cfg = os.path.join(experiment_path, "multirun.yaml")
    all_experiments_path = f'../logs/eval/ATATPeriodic{type}Comparison.csv'
    baseline_path = f'../logs/eval/{type}_Baselines.csv'
    with open(experiment_cfg, "r") as f:
        cfg = yaml.safe_load(f)

    cfg = OmegaConf.create(cfg)
    # Remove the date/time suffix from the experiment path
    experiment_name_path = "_".join(experiment_path.split('/')[-1].split('_')[:-2])
    os.makedirs(os.path.join('/home/fsoto/Documents/LCsSSL/logs/eval', experiment_name_path), exist_ok=True)
    cfg.paths.output_dir = os.path.join('/home/fsoto/Documents/LCsSSL/logs/eval', experiment_name_path)
    cfg.ckpt_path = experiment_path

    # Dynamically load classes and baseline metrics
    #cfg.classes = [
    #    'AGN', 'EB/EW', 'Blazar', 'CEP', 'LPV', 'CV/Nova', 'Periodic-Other', 'Microlensing', 'QSO',
    #    'DSCT', 'EA', 'RRLab', 'RSCVn', 'RRLc', 'SLSN', 'SNII', 'SNIbc', 'SNIIn', 'SNIa', 'TDE', 'YSO'
    #]
    #cfg.classes_order = [
    #    'SNIa', 'SNIbc', 'SNII', 'SNIIn', 'SLSN', 'TDE', 'Microlensing', 'QSO', 'AGN', 'Blazar', 'YSO',
    #    'CV/Nova', 'LPV', 'EA', 'EB/EW', 'Periodic-Other', 'RSCVn', 'CEP', 'RRLab', 'RRLc', 'DSCT'
    #]
    cfg.classes = ['CEP', 'DSCT', 'EA', 'EB/EW', 'LPV', 'Per-Other', 'RRLab', 'RRLc', 'RSCVn']
    cfg.classes_order = ['RRLc', 'RRLab', 'CEP', 'DSCT', 'EA', 'EB/EW', 'LPV','RSCVn', 'Per-Other']
    cfg.cm_title = cm_title
    cfg.trainer.devices = [0]  # Use only one GPU for evaluation
    cfg.experiment_name = experiment_name
    cfg.all_experiments_csv = all_experiments_path
    cfg.baseline_path = baseline_path
    cfg.baseline_experiment_name = baseline_name
    cfg = OmegaConf.merge(cfg, OmegaConf.from_dotlist([]))
    evaluate(cfg, test_type='test')

In [ ]:
experiments_lcs = [
    {   'type': 'lc',
        'name': 'ATAT_Periodic_Lightcurves_2025-10-02_16-57-36',
        'cm_title': 'ATAT LC-MTA',
        'experiment_name': 'ATAT LC',
        'baseline_name': 'ATAT LC'
    },
    {   'type': 'lc',
        'name': 'DiT_Periodic_Lightcurves_2025-10-21_14-40-20',
        'cm_title': 'DiffT LC-MTA',
        'experiment_name': 'DiffT LC',
        'baseline_name': 'ATAT LC'
    },
    {   'type': 'lc',
        'name': 'DiT_Periodic_VICReg_Lightcurves_2025-10-21_20-18-44',
        'cm_title': 'DiffT LC-MTA-PT',
        'experiment_name': 'DiffT_VICReg LC',
        'baseline_name': 'ATAT LC'
    },
    {   'type': 'lc',
        'name': 'DiT_Periodic_VICReg_LinearClassification_2025-10-22_12-28-42',
        'cm_title': 'DiffT LC-MTA-LP-PT',
        'experiment_name': 'DiffT_VICReg_Linear LC',
        'baseline_name': 'ATAT LC'
    },
    {   'type': 'lc',
        'name': 'DiViT_L_Periodic_Lightcurves_2025-10-24_15-33-41',
        'cm_title': 'DiffViT-L LC-MTA',
        'experiment_name': 'DiffViT_L LC',
        'baseline_name': 'ATAT LC'
    },
    {   'type': 'lc',
        'name': 'DiViT_L_Periodic_VICReg_Lightcurves_2025-10-24_20-42-35',
        'cm_title': 'DiffViT-L LC-MTA-PT',
        'experiment_name': 'DiffViT_L_VICReg LC',
        'baseline_name': 'ATAT LC'
    },
    {   'type': 'lc',
        'name': 'DiViT_L_Periodic_VICReg_LinearClassification_2025-10-25_11-08-43',
        'cm_title': 'DiffViT-L LC-MTA-LP-PT',
        'experiment_name': 'DiffViT_L_VICReg_Linear LC',
        'baseline_name': 'DiffT_VICReg_Linear LC'
    },
    {   'type': 'lc',
        'name': 'DiViT_Periodic_Lightcurves_2025-10-23_15-25-10',
        'cm_title': 'DiffViT LC-MTA',
        'experiment_name': 'DiffViT LC',
        'baseline_name': 'ATAT LC'
    },
    {   'type': 'lc',
        'name': 'DiViT_Periodic_VICReg_Lightcurves_2025-10-23_19-07-03',
        'cm_title': 'DiffViT LC-MTA-PT',
        'experiment_name': 'DiffViT_VICReg LC',
        'baseline_name': 'ATAT LC'
    },
    {   'type': 'lc',
        'name': 'DiViT_Periodic_VICReg_LinearClassification_2025-10-24_11-00-07',
        'cm_title': 'DiffViT LC-MTA-LP-PT',
        'experiment_name': 'DiffViT_VICReg_Linear LC',
        'baseline_name': 'DiffT_VICReg_Linear LC'
    }
]



In [ ]:
experiments_mm = [
    {   'type': 'multimodal',
        'name': 'ATAT_Periodic_MM_2025-10-09_14-01-07',
        'cm_title': 'ATAT LC-MD-FT-MTA',
        'experiment_name': 'ATAT MM',
        'baseline_name': 'ATAT MM'
    },
    {   'type': 'multimodal',
        'name': 'DiT_Periodic_MM_2025-10-09_18-22-05',
        'cm_title': 'DiffT LC-MD-FT-MTA',
        'experiment_name': 'DiffT MM',
        'baseline_name': 'ATAT MM'
    },
    {   'type': 'multimodal',
        'name': 'DiT_Periodic_VICReg_MM_2025-10-13_07-41-01',
        'cm_title': 'DiffT LC-MD-FT-MTA-PT',
        'experiment_name': 'DiffT_VICReg MM',
        'baseline_name': 'ATAT MM'
    },
    {   'type': 'multimodal',
        'name': 'DiViT_L_Periodic_MM_2025-10-14_04-39-51',
        'cm_title': 'DiffViT-L LC-MD-FT-MTA',
        'experiment_name': 'DiffViT_L MM',
        'baseline_name': 'ATAT MM'
    },
    {   'type': 'multimodal',
        'name': 'DiViT_L_Periodic_VICReg_MM_2025-10-14_07-18-37',
        'cm_title': 'DiffViT-L LC-MD-FT-MTA-PT',
        'experiment_name': 'DiffViT_L_VICReg MM',
        'baseline_name': 'ATAT MM'
    },
    {   'type': 'multimodal',
        'name': 'DiViT_Periodic_MM_2025-10-13_17-49-22',
        'cm_title': 'DiffViT LC-MD-FT-MTA',
        'experiment_name': 'DiffViT MM',
        'baseline_name': 'ATAT MM'
    },
    {   'type': 'multimodal',
        'name': 'DiViT_Periodic_VICReg_MM_2025-10-13_19-46-30',
        'cm_title': 'DiffViT LC-MD-FT-MTA-PT',
        'experiment_name': 'DiffViT_VICReg MM',
        'baseline_name': 'ATAT MM'
    }
]



In [ ]:
experiments_lcmd = [
    {   'type': 'lc_md',
        'name': 'ATAT_Periodic_LC_MD_2025-10-18_04-11-11',
        'cm_title': 'ATAT LC-MD-MTA',
        'experiment_name': 'ATAT LC_MD',
        'baseline_name': 'ATAT LC_MD'
    },
    {   'type': 'lc_md',
        'name': 'DiT_Periodic_LC_MD_2025-10-16_08-22-17',
        'cm_title': 'DiffT LC-MD-MTA',
        'experiment_name': 'DiffT LC_MD',
        'baseline_name': 'ATAT LC_MD'
    },
    {   'type': 'lc_md',
        'name': 'DiT_Periodic_VICReg_LC_MD_2025-10-16_20-26-18',
        'cm_title': 'DiffT LC-MD-MTA-PT',
        'experiment_name': 'DiffT_VICReg LC_MD',
        'baseline_name': 'ATAT LC_MD'
    },
    {   'type': 'lc_md',
        'name': 'DiViT_L_Periodic_LC_MD_2025-10-17_16-45-39',
        'cm_title': 'DiffViT-L LC-MD-MTA',
        'experiment_name': 'DiffViT_L LC_MD',
        'baseline_name': 'ATAT LC_MD'
    },
    {   'type': 'lc_md',
        'name': 'DiViT_L_Periodic_VICReg_LC_MD_2025-10-17_19-32-44',
        'cm_title': 'DiffViT-L LC-MD-MTA-PT',
        'experiment_name': 'DiffViT_L_VICReg LC_MD',
        'baseline_name': 'ATAT LC_MD'
    },
    {   'type': 'lc_md',
        'name': 'DiViT_Periodic_LC_MD_2025-10-17_05-55-04',
        'cm_title': 'DiffViT LC-MD-MTA',
        'experiment_name': 'DiffViT LC_MD',
        'baseline_name': 'ATAT LC_MD'
    },
    {   'type': 'lc_md',
        'name': 'DiViT_Periodic_VICReg_LC_MD_2025-10-17_07-55-48',
        'cm_title': 'DiffViT LC-MD-MTA-PT',
        'experiment_name': 'DiffViT_VICReg LC_MD',
        'baseline_name': 'ATAT LC_MD'
    }
]



In [8]:
for exp in experiments_lcs:
    eval_several_exp( exp_dict=exp)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loaded model weights from the checkpoint at ../logs/lc/DiViT_L_Periodic_Lightcurves_2025-10-24_15-33-41/24/checkpoints/epoch_0151.ckpt
Loaded model weights from the checkpoint at ../logs/lc/DiViT_L_Periodic_Lightcurves_2025-10-24_15-33-41/24/checkpoints/epoch_0151.ckpt


Predicting DataLoader 0: 100%|██████████| 36/36 [00:01<00:00, 34.81it/s]

Metrics saved to: ../logs/eval/ATATPeriodiclcComparison_LC.csv
Metrics saved to: ../logs/eval/ATATPeriodiclcComparison_LC.csv
Uploaded ../logs/eval/ATATPeriodiclcComparison_LC.csv to sheet: ATATPeriodiclcComparison_LC
Uploaded ../logs/eval/ATATPeriodiclcComparison_LC.csv to sheet: ATATPeriodiclcComparison_LC


Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


25 25
Evaluating split 0 with checkpoint ../logs/lc/DiViT_L_Periodic_VICReg_Lightcurves_2025-10-24_20-42-35/14/checkpoints/epoch_0078.ckpt
torch.Size([1, 199, 128]) torch.Size([1, 199, 128])
Loading pre-trained weights from /home/fsoto/Documents/LCsSSL/logs/pre_trained_models/VICReg_divit_L/1/checkpoints/epoch_0186.ckpt
Checkpoint keys: ['network.transformer_lc.stacked_encoders.0.norm1.weight', 'network.transformer_lc.stacked_encoders.0.norm1.bias', 'network.transformer_lc.stacked_encoders.0.norm2.weight', 'network.transformer_lc.stacked_encoders.0.norm2.bias', 'network.transformer_lc.stacked_encoders.0.attn.qkv_proj.weight']
Model has 61 parameters
Checkpoint has 61 parameters for network
Sample model keys: ['time_encoder.time_encoders.0.rate_emb.0.weight', 'transformer_lc.stacked_encoders.1.attn.out_proj.weight', 'time_encoder.time_encoders.0.fusion_mlp.6.bias', 'time_encoder.time_encoders.0.rate_emb.3.bias', 'time_encoder.time_encoders.0.fusion_mlp.2.bias']
Sample checkpoint keys: [

Restoring states from the checkpoint path at ../logs/lc/DiViT_L_Periodic_VICReg_Lightcurves_2025-10-24_20-42-35/14/checkpoints/epoch_0078.ckpt


...

Checkpoint keys: ['network.transformer_lc.stacked_encoders.0.norm1.weight', 'network.transformer_lc.stacked_encoders.0.norm1.bias', 'network.transformer_lc.stacked_encoders.0.norm2.weight', 'network.transformer_lc.stacked_encoders.0.norm2.bias', 'network.transformer_lc.stacked_encoders.0.attn.qkv_proj.weight']
Model has 61 parameters
Checkpoint has 61 parameters for network
Sample model keys: ['time_encoder.time_encoders.0.rate_emb.0.weight', 'transformer_lc.stacked_encoders.1.attn.out_proj.weight', 'time_encoder.time_encoders.0.fusion_mlp.6.bias', 'time_encoder.time_encoders.0.rate_emb.3.bias', 'time_encoder.time_encoders.0.fusion_mlp.2.bias']
Sample checkpoint keys: ['time_encoder.time_encoders.0.rate_emb.0.weight', 'transformer_lc.stacked_encoders.1.attn.out_proj.weight', 'time_encoder.time_encoders.0.fusion_mlp.6.bias', 'time_encoder.time_encoders.0.rate_emb.3.bias', 'time_encoder.time_encoders.0.fusion_mlp.2.bias']
✓ Successfully loaded lightcurve model from /home/fsoto/Documents/

Restoring states from the checkpoint path at ../logs/lc/DiViT_Periodic_VICReg_LinearClassification_2025-10-24_11-00-07/23/checkpoints/epoch_0158.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loaded model weights from the checkpoint at ../logs/lc/DiViT_Periodic_VICReg_LinearClassification_2025-10-24_11-00-07/23/checkpoints/epoch_0158.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loaded model weights from the checkpoint at ../logs/lc/DiViT_Periodic_VICReg_LinearClassification_2025-10-24_11-00-07/23/checkpoints/epoch_0158.ckpt


Checkpoint keys: ['network.transformer_lc.stacked_encoders.0.norm1.weight', 'network.transformer_lc.stacked_encoders.0.norm1.bias', 'network.transformer_lc.stacked_encoders.0.norm2.weight', 'network.transformer_lc.stacked_encoders.0.norm2.bias', 'network.transformer_lc.stacked_encoders.0.attn.qkv_proj.weight']
Model has 61 parameters
Checkpoint has 61 parameters for network
Sample model keys: ['time_encoder.time_encoders.0.rate_emb.0.weight', 'transformer_lc.stacked_encoders.1.attn.out_proj.weight', 'time_encoder.time_encoders.0.fusion_mlp.6.bias', 'time_encoder.time_encoders.0.rate_emb.3.bias', 'time_encoder.time_encoders.0.fusion_mlp.2.bias']
Sample checkpoint keys: ['time_encoder.time_encoders.0.rate_emb.0.weight', 'transformer_lc.stacked_encoders.1.attn.out_proj.weight', 'time_encoder.time_encoders.0.fusion_mlp.6.bias', 'time_encoder.time_encoders.0.rate_emb.3.bias', 'time_encoder.time_encoders.0.fusion_mlp.2.bias']
✓ Successfully loaded lightcurve model from /home/fsoto/Documents/

Restoring states from the checkpoint path at ../logs/lc/DiViT_Periodic_VICReg_LinearClassification_2025-10-24_11-00-07/24/checkpoints/epoch_0158.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loaded model weights from the checkpoint at ../logs/lc/DiViT_Periodic_VICReg_LinearClassification_2025-10-24_11-00-07/24/checkpoints/epoch_0158.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loaded model weights from the checkpoint at ../logs/lc/DiViT_Periodic_VICReg_LinearClassification_2025-10-24_11-00-07/24/checkpoints/epoch_0158.ckpt


Checkpoint keys: ['network.transformer_lc.stacked_encoders.0.norm1.weight', 'network.transformer_lc.stacked_encoders.0.norm1.bias', 'network.transformer_lc.stacked_encoders.0.norm2.weight', 'network.transformer_lc.stacked_encoders.0.norm2.bias', 'network.transformer_lc.stacked_encoders.0.attn.qkv_proj.weight']
Model has 61 parameters
Checkpoint has 61 parameters for network
Sample model keys: ['time_encoder.time_encoders.0.rate_emb.0.weight', 'transformer_lc.stacked_encoders.1.attn.out_proj.weight', 'time_encoder.time_encoders.0.fusion_mlp.6.bias', 'time_encoder.time_encoders.0.rate_emb.3.bias', 'time_encoder.time_encoders.0.fusion_mlp.2.bias']
Sample checkpoint keys: ['time_encoder.time_encoders.0.rate_emb.0.weight', 'transformer_lc.stacked_encoders.1.attn.out_proj.weight', 'time_encoder.time_encoders.0.fusion_mlp.6.bias', 'time_encoder.time_encoders.0.rate_emb.3.bias', 'time_encoder.time_encoders.0.fusion_mlp.2.bias']
✓ Successfully loaded lightcurve model from /home/fsoto/Documents/

In [9]:
for exp in experiments_lcmd:
    eval_several_exp( exp_dict=exp)

Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


5 5
Evaluating split 0 with checkpoint ../logs/lc_md/ATAT_Periodic_LC_MD_2025-10-18_04-11-11/0/checkpoints/epoch_0076.ckpt
torch.Size([1, 8, 128]) torch.Size([1, 8, 128])

Model Complexity:
FLOPs: 253.04G
Params: 1.57M

Model Complexity:
FLOPs: 253.04G
Params: 1.57M


Restoring states from the checkpoint path at ../logs/lc_md/ATAT_Periodic_LC_MD_2025-10-18_04-11-11/0/checkpoints/epoch_0076.ckpt


Average Inference Time: 51.35ms

✓ Model complexity saved to: /home/fsoto/Documents/LCsSSL/logs/eval/ATAT_Periodic_LC_MD/model_complexity.txt

torch.Size([1, 8, 128]) torch.Size([1, 8, 128])


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loaded model weights from the checkpoint at ../logs/lc_md/ATAT_Periodic_LC_MD_2025-10-18_04-11-11/0/checkpoints/epoch_0076.ckpt
Loaded model weights from the checkpoint at ../logs/lc_md/ATAT_Periodic_LC_MD_2025-10-18_04-11-11/0/checkpoints/epoch_0076.ckpt


...

Checkpoint keys: ['network.transformer_lc.stacked_encoders.0.norm1.weight', 'network.transformer_lc.stacked_encoders.0.norm1.bias', 'network.transformer_lc.stacked_encoders.0.norm2.weight', 'network.transformer_lc.stacked_encoders.0.norm2.bias', 'network.transformer_lc.stacked_encoders.0.attn.qkv_proj.weight']
Model has 61 parameters
Checkpoint has 61 parameters for network
Sample model keys: ['time_encoder.time_encoders.0.rate_emb.0.weight', 'transformer_lc.stacked_encoders.1.attn.out_proj.weight', 'time_encoder.time_encoders.0.fusion_mlp.6.bias', 'time_encoder.time_encoders.0.rate_emb.3.bias', 'time_encoder.time_encoders.0.fusion_mlp.2.bias']
Sample checkpoint keys: ['time_encoder.time_encoders.0.rate_emb.0.weight', 'transformer_lc.stacked_encoders.1.attn.out_proj.weight', 'time_encoder.time_encoders.0.fusion_mlp.6.bias', 'time_encoder.time_encoders.0.rate_emb.3.bias', 'time_encoder.time_encoders.0.fusion_mlp.2.bias']
✓ Successfully loaded lightcurve model from /home/fsoto/Documents/

Restoring states from the checkpoint path at ../logs/lc_md/DiViT_Periodic_VICReg_LC_MD_2025-10-17_07-55-48/23/checkpoints/epoch_0172.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loaded model weights from the checkpoint at ../logs/lc_md/DiViT_Periodic_VICReg_LC_MD_2025-10-17_07-55-48/23/checkpoints/epoch_0172.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loaded model weights from the checkpoint at ../logs/lc_md/DiViT_Periodic_VICReg_LC_MD_2025-10-17_07-55-48/23/checkpoints/epoch_0172.ckpt


Checkpoint keys: ['network.transformer_lc.stacked_encoders.0.norm1.weight', 'network.transformer_lc.stacked_encoders.0.norm1.bias', 'network.transformer_lc.stacked_encoders.0.norm2.weight', 'network.transformer_lc.stacked_encoders.0.norm2.bias', 'network.transformer_lc.stacked_encoders.0.attn.qkv_proj.weight']
Model has 61 parameters
Checkpoint has 61 parameters for network
Sample model keys: ['time_encoder.time_encoders.0.rate_emb.0.weight', 'transformer_lc.stacked_encoders.1.attn.out_proj.weight', 'time_encoder.time_encoders.0.fusion_mlp.6.bias', 'time_encoder.time_encoders.0.rate_emb.3.bias', 'time_encoder.time_encoders.0.fusion_mlp.2.bias']
Sample checkpoint keys: ['time_encoder.time_encoders.0.rate_emb.0.weight', 'transformer_lc.stacked_encoders.1.attn.out_proj.weight', 'time_encoder.time_encoders.0.fusion_mlp.6.bias', 'time_encoder.time_encoders.0.rate_emb.3.bias', 'time_encoder.time_encoders.0.fusion_mlp.2.bias']
✓ Successfully loaded lightcurve model from /home/fsoto/Documents/

Restoring states from the checkpoint path at ../logs/lc_md/DiViT_Periodic_VICReg_LC_MD_2025-10-17_07-55-48/24/checkpoints/epoch_0098.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loaded model weights from the checkpoint at ../logs/lc_md/DiViT_Periodic_VICReg_LC_MD_2025-10-17_07-55-48/24/checkpoints/epoch_0098.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loaded model weights from the checkpoint at ../logs/lc_md/DiViT_Periodic_VICReg_LC_MD_2025-10-17_07-55-48/24/checkpoints/epoch_0098.ckpt


Checkpoint keys: ['network.transformer_lc.stacked_encoders.0.norm1.weight', 'network.transformer_lc.stacked_encoders.0.norm1.bias', 'network.transformer_lc.stacked_encoders.0.norm2.weight', 'network.transformer_lc.stacked_encoders.0.norm2.bias', 'network.transformer_lc.stacked_encoders.0.attn.qkv_proj.weight']
Model has 61 parameters
Checkpoint has 61 parameters for network
Sample model keys: ['time_encoder.time_encoders.0.rate_emb.0.weight', 'transformer_lc.stacked_encoders.1.attn.out_proj.weight', 'time_encoder.time_encoders.0.fusion_mlp.6.bias', 'time_encoder.time_encoders.0.rate_emb.3.bias', 'time_encoder.time_encoders.0.fusion_mlp.2.bias']
Sample checkpoint keys: ['time_encoder.time_encoders.0.rate_emb.0.weight', 'transformer_lc.stacked_encoders.1.attn.out_proj.weight', 'time_encoder.time_encoders.0.fusion_mlp.6.bias', 'time_encoder.time_encoders.0.rate_emb.3.bias', 'time_encoder.time_encoders.0.fusion_mlp.2.bias']
✓ Successfully loaded lightcurve model from /home/fsoto/Documents/

In [10]:
for exp in experiments_mm:
    eval_several_exp( exp_dict=exp)

Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


5 5
Evaluating split 0 with checkpoint ../logs/multimodal/ATAT_Periodic_MM_2025-10-09_14-01-07/0/checkpoints/epoch_0098.ckpt
torch.Size([1, 199, 128]) torch.Size([1, 199, 128])

Model Complexity:
FLOPs: 262.73G
Params: 1.57M

Model Complexity:
FLOPs: 262.73G
Params: 1.57M
Average Inference Time: 51.02ms

✓ Model complexity saved to: /home/fsoto/Documents/LCsSSL/logs/eval/ATAT_Periodic_MM/model_complexity.txt

torch.Size([1, 199, 128]) torch.Size([1, 199, 128])
Average Inference Time: 51.02ms

✓ Model complexity saved to: /home/fsoto/Documents/LCsSSL/logs/eval/ATAT_Periodic_MM/model_complexity.txt

torch.Size([1, 199, 128]) torch.Size([1, 199, 128])


Restoring states from the checkpoint path at ../logs/multimodal/ATAT_Periodic_MM_2025-10-09_14-01-07/0/checkpoints/epoch_0098.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loaded model weights from the checkpoint at ../logs/multimodal/ATAT_Periodic_MM_2025-10-09_14-01-07/0/checkpoints/epoch_0098.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loaded model weights from the checkpoint at ../logs/multimodal/ATAT_Periodic_MM_2025-10-09_14-01-07/0/checkpoints/epoch_0098.ckpt


Predicting DataLoader 0: 100%|██████████| 36/36 [00:01<00:00, 22.98it/s]

Evaluating split 1 with checkpoint ../logs/multimodal/ATAT_Periodic_MM_2025-10-09_14-01-07/1/checkpoints/epoch_0089.ckpt
torch.Size([1, 199, 128]) torch.Size([1, 199, 128])
torch.Size([1, 199, 128]) torch.Size([1, 199, 128])
Evaluating split 1 with checkpoint ../logs/multimodal/ATAT_Periodic_MM_2025-10-09_14-01-07/1/checkpoints/epoch_0089.ckpt
torch.Size([1, 199, 128]) torch.Size([1, 199, 128])
torch.Size([1, 199, 128]) torch.Size([1, 199, 128])


Restoring states from the checkpoint path at ../logs/multimodal/ATAT_Periodic_MM_2025-10-09_14-01-07/1/checkpoints/epoch_0089.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loaded model weights from the checkpoint at ../logs/multimodal/ATAT_Periodic_MM_2025-10-09_14-01-07/1/checkpoints/epoch_0089.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loaded model weights from the checkpoint at ../logs/multimodal/ATAT_Periodic_MM_2025-10-09_14-01-07/1/checkpoints/epoch_0089.ckpt


...

Predicting DataLoader 0: 100%|██████████| 36/36 [00:01<00:00, 35.85it/s]

Evaluating split 23 with checkpoint ../logs/multimodal/DiViT_Periodic_VICReg_MM_2025-10-13_19-46-30/23/checkpoints/epoch_0165.ckpt
torch.Size([1, 199, 128]) torch.Size([1, 199, 128])
Loading pre-trained weights from /home/fsoto/Documents/LCsSSL/logs/pre_trained_models/VICReg_divit/1/checkpoints/epoch_0285.ckpt
Evaluating split 23 with checkpoint ../logs/multimodal/DiViT_Periodic_VICReg_MM_2025-10-13_19-46-30/23/checkpoints/epoch_0165.ckpt
torch.Size([1, 199, 128]) torch.Size([1, 199, 128])
Loading pre-trained weights from /home/fsoto/Documents/LCsSSL/logs/pre_trained_models/VICReg_divit/1/checkpoints/epoch_0285.ckpt
Checkpoint keys: ['network.transformer_lc.stacked_encoders.0.norm1.weight', 'network.transformer_lc.stacked_encoders.0.norm1.bias', 'network.transformer_lc.stacked_encoders.0.norm2.weight', 'network.transformer_lc.stacked_encoders.0.norm2.bias', 'network.transformer_lc.stacked_encoders.0.attn.qkv_proj

Restoring states from the checkpoint path at ../logs/multimodal/DiViT_Periodic_VICReg_MM_2025-10-13_19-46-30/23/checkpoints/epoch_0165.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loaded model weights from the checkpoint at ../logs/multimodal/DiViT_Periodic_VICReg_MM_2025-10-13_19-46-30/23/checkpoints/epoch_0165.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loaded model weights from the checkpoint at ../logs/multimodal/DiViT_Periodic_VICReg_MM_2025-10-13_19-46-30/23/checkpoints/epoch_0165.ckpt


Predicting DataLoader 0: 100%|██████████| 36/36 [00:01<00:00, 34.31it/s]

Evaluating split 24 with checkpoint ../logs/multimodal/DiViT_Periodic_VICReg_MM_2025-10-13_19-46-30/24/checkpoints/epoch_0165.ckpt
torch.Size([1, 199, 128]) torch.Size([1, 199, 128])
Loading pre-trained weights from /home/fsoto/Documents/LCsSSL/logs/pre_trained_models/VICReg_divit/1/checkpoints/epoch_0285.ckpt
Evaluating split 24 with checkpoint ../logs/multimodal/DiViT_Periodic_VICReg_MM_2025-10-13_19-46-30/24/checkpoints/epoch_0165.ckpt
torch.Size([1, 199, 128]) torch.Size([1, 199, 128])
Loading pre-trained weights from /home/fsoto/Documents/LCsSSL/logs/pre_trained_models/VICReg_divit/1/checkpoints/epoch_0285.ckpt
Checkpoint keys: ['network.transformer_lc.stacked_encoders.0.norm1.weight', 'network.transformer_lc.stacked_encoders.0.norm1.bias', 'network.transformer_lc.stacked_encoders.0.norm2.weight', 'network.transformer_lc.stacked_encoders.0.norm2.bias', 'network.transformer_lc.stacked_encoders.0.attn.qkv_proj

Restoring states from the checkpoint path at ../logs/multimodal/DiViT_Periodic_VICReg_MM_2025-10-13_19-46-30/24/checkpoints/epoch_0165.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loaded model weights from the checkpoint at ../logs/multimodal/DiViT_Periodic_VICReg_MM_2025-10-13_19-46-30/24/checkpoints/epoch_0165.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loaded model weights from the checkpoint at ../logs/multimodal/DiViT_Periodic_VICReg_MM_2025-10-13_19-46-30/24/checkpoints/epoch_0165.ckpt


Predicting DataLoader 0: 100%|██████████| 36/36 [00:00<00:00, 37.75it/s]

Metrics saved to: ../logs/eval/ATATPeriodicmultimodalComparison_LC.csv
Metrics saved to: ../logs/eval/ATATPeriodicmultimodalComparison_LC.csv
Uploaded ../logs/eval/ATATPeriodicmultimodalComparison_LC.csv to sheet: ATATPeriodicmultimodalComparison_LC
Uploaded ../logs/eval/ATATPeriodicmultimodalComparison_LC.csv to sheet: ATATPeriodicmultimodalComparison_LC
Metrics saved to: ../logs/eval/ATATPeriodicmultimodalComparison_Feat.csv
Metrics saved to: ../logs/eval/ATATPeriodicmultimodalComparison_Feat.csv
Uploaded ../logs/eval/ATATPeriodicmultimodalComparison_Feat.csv to sheet: ATATPeriodicmultimodalComparison_Feat
Uploaded ../logs/eval/ATATPeriodicmultimodalComparison_Feat.csv to sheet: ATATPeriodicmultimodalComparison_Feat
Metrics saved to: ../logs/eval/ATATPeriodicmultimodalComparison_Mix.csv
Metrics saved to: ../logs/eval/ATATPeriodicmultimodalComparison_Mix.csv
Uploaded ../logs/eval/ATATPeriodicmultimodalComparison